In [1]:
from pathlib import Path
from typing import cast
import re

import numpy as np
import librosa
import jams
from jams import Annotation

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "guitarset"
AUDIO_DIR = DATA_DIR / "audio_mono-mic"
ANNO_DIR = DATA_DIR / "annotation"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SR = 22050              # target sample rate
WINDOW_SEC = 0.5        # length of each training window
WINDOW_SAMPLES = int(SR * WINDOW_SEC)

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed dir: {PROCESSED_DIR}")
print(f"Window: {WINDOW_SEC}s = {WINDOW_SAMPLES} samples at {SR} Hz")

Project root: c:\Users\Constantine\Code\ml-project
Processed dir: c:\Users\Constantine\Code\ml-project\data\processed
Window: 0.5s = 11025 samples at 22050 Hz


In [2]:
# 12 pitch classes, sharp notation (Harte-standard)
PITCH_CLASSES = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]

# Build the 37-class label list
# 0-11:  single notes    (e.g. "note:C", "note:C#", ...)
# 12-23: major triads    (e.g. "maj:C", "maj:C#", ...)
# 24-35: minor triads    (e.g. "min:C", "min:C#", ...)
# 36:    silence
CLASS_NAMES: list[str] = (
    [f"note:{p}" for p in PITCH_CLASSES]
    + [f"maj:{p}" for p in PITCH_CLASSES]
    + [f"min:{p}" for p in PITCH_CLASSES]
    + ["silence"]
)

NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_ID: dict[str, int] = {name: i for i, name in enumerate(CLASS_NAMES)}
ID_TO_CLASS: dict[int, str] = {i: name for i, name in enumerate(CLASS_NAMES)}

# Sanity check
print(f"Total classes: {NUM_CLASSES}")
print(f"First 3 classes: {CLASS_NAMES[:3]}")
print(f"Classes 12-14:   {CLASS_NAMES[12:15]}")
print(f"Classes 24-26:   {CLASS_NAMES[24:27]}")
print(f"Last class:      {CLASS_NAMES[-1]}")

Total classes: 37
First 3 classes: ['note:C', 'note:C#', 'note:D']
Classes 12-14:   ['maj:C', 'maj:C#', 'maj:D']
Classes 24-26:   ['min:C', 'min:C#', 'min:D']
Last class:      silence


In [3]:
# Enharmonic equivalents — GuitarSet uses both sharps and flats
FLAT_TO_SHARP = {
    "Cb": "B", "Db": "C#", "Eb": "D#", "Fb": "E",
    "Gb": "F#", "Ab": "G#", "Bb": "A#",
}

# Chord quality suffixes that collapse to "major triad"
# Any 7/9/11/13 extension of a major triad, and add chords
MAJOR_QUALITIES = {"maj", "maj6", "maj7", "maj9", "maj11", "maj13",
                   "7", "9", "11", "13", "add9"}

# Chord quality suffixes that collapse to "minor triad"
MINOR_QUALITIES = {"min", "min6", "min7", "min9", "min11", "min13",
                   "minmaj7"}

# Everything else we skip (sus, dim, aug, hdim, etc.)


def normalize_pitch(pitch: str) -> str:
    """Convert flat spellings to sharp spellings. B# → C, etc."""
    if pitch in FLAT_TO_SHARP:
        return FLAT_TO_SHARP[pitch]
    if pitch == "B#":
        return "C"
    if pitch == "E#":
        return "F"
    return pitch


def harte_to_class_id(chord_label: str) -> int | None:
    """
    Convert a Harte-format chord label to our class ID (12-35), or None if
    we can't cleanly place it into major/minor.

    Returns:
        int  — class ID (12-35 for chords, 36 for silence)
        None — chord should be skipped (ambiguous or unsupported quality)
    """
    # Silence / no-chord
    if chord_label in ("N", "No", "silence"):
        return CLASS_TO_ID["silence"]
    if chord_label == "X":
        return None

    # Strip bass note (inversions): "C:maj/3" → "C:maj"
    if "/" in chord_label:
        chord_label = chord_label.split("/")[0]

    # Split root and quality
    if ":" not in chord_label:
        # Bare pitch means major triad by Harte convention: "C" == "C:maj"
        root = chord_label
        quality = "maj"
    else:
        root, quality = chord_label.split(":", 1)

    root = normalize_pitch(root)
    if root not in PITCH_CLASSES:
        return None  # Unrecognized root

    # Strip any explicit interval list: "C:maj(9)" → "C:maj"
    quality = re.sub(r"\(.*?\)", "", quality).strip()

    if quality in MAJOR_QUALITIES:
        return CLASS_TO_ID[f"maj:{root}"]
    if quality in MINOR_QUALITIES:
        return CLASS_TO_ID[f"min:{root}"]

    # Everything else — sus, dim, aug, hdim, unknown — we skip
    return None


# Test on our familiar file's chord labels + some tricky ones
test_labels = [
    "D#:maj", "G#:maj", "A#:maj",      # from our Eb Bossa Nova file
    "Eb:maj", "Ab:maj",                # flat equivalents (should map to same class)
    "C:maj", "A:min",                  # basics
    "G:7", "D:min7", "F:maj7",         # extensions
    "C:sus4", "F#:dim", "A:aug",       # should skip
    "C", "N", "X",                     # bare pitch, silence, unknown
    "C:maj/3", "G:7/5",                # inversions
]

print(f"{'Input':>12}  →  {'Class ID':>8}  {'Class name'}")
print("-" * 50)
for lbl in test_labels:
    cid = harte_to_class_id(lbl)
    name = ID_TO_CLASS[cid] if cid is not None else "(skipped)"
    print(f"{lbl:>12}  →  {str(cid):>8}  {name}")

       Input  →  Class ID  Class name
--------------------------------------------------
      D#:maj  →        15  maj:D#
      G#:maj  →        20  maj:G#
      A#:maj  →        22  maj:A#
      Eb:maj  →        15  maj:D#
      Ab:maj  →        20  maj:G#
       C:maj  →        12  maj:C
       A:min  →        33  min:A
         G:7  →        19  maj:G
      D:min7  →        26  min:D
      F:maj7  →        17  maj:F
      C:sus4  →      None  (skipped)
      F#:dim  →      None  (skipped)
       A:aug  →      None  (skipped)
           C  →        12  maj:C
           N  →        36  silence
           X  →      None  (skipped)
     C:maj/3  →        12  maj:C
       G:7/5  →        19  maj:G


In [4]:
def extract_chord_segments(jam: jams.JAMS) -> list[tuple[float, float, int]]:
    """
    Extract chord segments from a JAMS file, converted to our class IDs.

    Returns:
        List of (start_sec, end_sec, class_id) tuples.
        Skipped chords (sus, dim, aug, unknown) are omitted entirely.
    """
    chord_anns = jam.search(namespace="chord")
    if not chord_anns:
        return []

    chord_ann = cast(Annotation, chord_anns[0])
    segments: list[tuple[float, float, int]] = []

    for obs in chord_ann.data:
        cid = harte_to_class_id(str(obs.value))
        if cid is None:
            continue  # skip sus, dim, aug, unknown
        start = float(obs.time)
        end = start + float(obs.duration)
        segments.append((start, end, cid))

    return segments


# Test on our familiar file
FILE_ID = "00_BN1-129-Eb_comp"
anno_path = ANNO_DIR / f"{FILE_ID}.jams"
jam = jams.load(str(anno_path))

segments = extract_chord_segments(jam)

print(f"Extracted {len(segments)} chord segments from {FILE_ID}\n")
print(f"{'start':>7}  {'end':>7}  {'dur':>6}  {'class_id':>8}  {'class_name'}")
print("-" * 50)
for start, end, cid in segments:
    print(f"{start:>7.3f}  {end:>7.3f}  {end - start:>6.3f}  {cid:>8}  {ID_TO_CLASS[cid]}")

Extracted 6 chord segments from 00_BN1-129-Eb_comp

  start      end     dur  class_id  class_name
--------------------------------------------------
  0.000    7.442   7.442        15  maj:D#
  7.442   11.163   3.721        20  maj:G#
 11.163   14.884   3.721        15  maj:D#
 14.884   16.744   1.860        22  maj:A#
 16.744   18.605   1.860        20  maj:G#
 18.605   22.324   3.720        15  maj:D#


In [5]:
def midi_to_pitch_class(midi_note: int) -> str:
    """MIDI note number → pitch class name. 60 → 'C', 61 → 'C#', ..."""
    return PITCH_CLASSES[midi_note % 12]


def extract_note_segments(
    jam: jams.JAMS,
    min_duration: float = 0.15,
    max_duration: float = 2.0,
) -> list[tuple[float, float, int]]:
    """
    Extract monophonic note segments — moments where exactly one note is sounding.

    Iterates all per-string note events; keeps a note only if no other string's
    note overlaps its time range (fully monophonic segment).

    Args:
        jam: Loaded JAMS annotation
        min_duration: Ignore notes shorter than this (transients, artifacts)
        max_duration: Ignore notes longer than this (held drones)

    Returns:
        List of (start_sec, end_sec, class_id) tuples for note classes (0-11).
    """
    note_anns = jam.search(namespace="note_midi")
    if not note_anns:
        return []

    # Collect ALL note events across all strings
    # Each: (start, end, midi_number, string_index)
    all_notes: list[tuple[float, float, int, int]] = []
    for string_idx, ann_raw in enumerate(note_anns):
        ann = cast(Annotation, ann_raw)
        for obs in ann.data:
            start = float(obs.time)
            end = start + float(obs.duration)
            midi = int(round(float(obs.value)))
            all_notes.append((start, end, midi, string_idx))

    monophonic_segments: list[tuple[float, float, int]] = []

    for start, end, midi, string_idx in all_notes:
        duration = end - start
        if duration < min_duration or duration > max_duration:
            continue

        # Check for temporal overlap with any note on OTHER strings
        overlaps_other = False
        for o_start, o_end, _o_midi, o_string in all_notes:
            if o_string == string_idx:
                continue  # same string is fine (sequential notes)
            # Overlap test: two intervals overlap iff neither ends before the other starts
            if o_start < end and o_end > start:
                overlaps_other = True
                break

        if overlaps_other:
            continue  # not monophonic at this moment

        pitch = midi_to_pitch_class(midi)
        cid = CLASS_TO_ID[f"note:{pitch}"]
        monophonic_segments.append((start, end, cid))

    return monophonic_segments


# Test on a solo file from same player + style
SOLO_FILE_ID = "00_BN1-129-Eb_solo"
solo_anno_path = ANNO_DIR / f"{SOLO_FILE_ID}.jams"
solo_jam = jams.load(str(solo_anno_path))

note_segments = extract_note_segments(solo_jam)

print(f"Extracted {len(note_segments)} monophonic note segments from {SOLO_FILE_ID}\n")
print("First 10:")
print(f"{'start':>7}  {'end':>7}  {'dur':>6}  {'class_id':>8}  {'class_name'}")
print("-" * 50)
for start, end, cid in note_segments[:10]:
    print(f"{start:>7.3f}  {end:>7.3f}  {end - start:>6.3f}  {cid:>8}  {ID_TO_CLASS[cid]}")

# Class distribution across this file
from collections import Counter
class_counts = Counter(ID_TO_CLASS[cid] for _, _, cid in note_segments)
print(f"\nClass distribution in {SOLO_FILE_ID}:")
for name, cnt in class_counts.most_common():
    print(f"  {name}: {cnt}")

Extracted 16 monophonic note segments from 00_BN1-129-Eb_solo

First 10:
  start      end     dur  class_id  class_name
--------------------------------------------------
  1.027    1.434   0.406         5  note:F
 19.598   20.086   0.488         5  note:F
 20.561   22.324   1.764         7  note:G
 13.530   14.186   0.656        10  note:A#
 14.229   14.467   0.238        10  note:A#
 14.897   15.106   0.209        10  note:A#
 15.112   15.390   0.279         9  note:A
 15.393   15.584   0.192         9  note:A
 17.698   17.912   0.215        10  note:A#
 17.915   18.188   0.273         9  note:A

Class distribution in 00_BN1-129-Eb_solo:
  note:A#: 6
  note:A: 4
  note:F: 2
  note:C: 2
  note:G: 1
  note:D: 1


In [6]:
HOP_SEC = 0.25  # 50% overlap between windows
HOP_SAMPLES = int(SR * HOP_SEC)


def cut_windows_from_segment(
    audio: np.ndarray,
    start_sec: float,
    end_sec: float,
    class_id: int,
) -> list[tuple[np.ndarray, int]]:
    """
    Cut a labeled time range into fixed-length overlapping windows.

    Args:
        audio: Full audio array (1D, at SR Hz)
        start_sec, end_sec: Segment boundaries in seconds
        class_id: Label for every window from this segment

    Returns:
        List of (window_audio, class_id) tuples. Empty if segment too short.
    """
    start_sample = int(start_sec * SR)
    end_sample = int(end_sec * SR)

    # Skip segments shorter than one window
    if end_sample - start_sample < WINDOW_SAMPLES:
        return []

    # Clip to audio bounds (in case annotation runs past the file)
    end_sample = min(end_sample, len(audio))

    windows: list[tuple[np.ndarray, int]] = []
    pos = start_sample
    while pos + WINDOW_SAMPLES <= end_sample:
        window = audio[pos : pos + WINDOW_SAMPLES]
        windows.append((window.copy(), class_id))
        pos += HOP_SAMPLES

    return windows


# Test on the comp file — load its audio and cut all chord segments into windows
audio_path = AUDIO_DIR / f"{FILE_ID}_mic.wav"
y, _ = librosa.load(audio_path, sr=SR, mono=True)

all_windows: list[tuple[np.ndarray, int]] = []
for start, end, cid in segments:  # from the chord segment extraction cell
    windows = cut_windows_from_segment(y, start, end, cid)
    all_windows.extend(windows)

print(f"Total windows cut from {FILE_ID}: {len(all_windows)}\n")

# Class distribution
from collections import Counter
window_counts = Counter(ID_TO_CLASS[cid] for _, cid in all_windows)
print("Windows per class:")
for name, cnt in window_counts.most_common():
    print(f"  {name}: {cnt}")

# Sanity: first window's shape and dtype
first_window, first_cid = all_windows[0]
print(f"\nFirst window: shape={first_window.shape}, dtype={first_window.dtype}")
print(f"First window label: class_id={first_cid} ({ID_TO_CLASS[first_cid]})")

Total windows cut from 00_BN1-129-Eb_comp: 79

Windows per class:
  maj:D#: 54
  maj:G#: 19
  maj:A#: 6

First window: shape=(11025,), dtype=float32
First window label: class_id=15 (maj:D#)


In [7]:
from tqdm import tqdm

def parse_file_id(audio_filename: str) -> tuple[str, str, str]:
    """
    Extract (file_id, player_id, mode) from an audio filename.
    e.g. '00_BN1-129-Eb_comp_mic.wav' → ('00_BN1-129-Eb_comp', '00', 'comp')
    """
    stem = audio_filename.replace("_mic.wav", "")
    player_id = stem.split("_")[0]                    # '00'
    mode = "comp" if stem.endswith("_comp") else "solo"
    return stem, player_id, mode


def process_one_file(
    audio_path: Path,
) -> list[tuple[np.ndarray, int, str, str]]:
    """
    Load one audio file, extract its labeled segments, cut into windows.
    Returns list of (window, class_id, file_id, player_id) tuples.
    """
    file_id, player_id, mode = parse_file_id(audio_path.name)
    anno_path = ANNO_DIR / f"{file_id}.jams"

    if not anno_path.exists():
        return []

    y, _ = librosa.load(audio_path, sr=SR, mono=True)
    jam = jams.load(str(anno_path))

    if mode == "comp":
        segments = extract_chord_segments(jam)
    else:
        segments = extract_note_segments(jam)

    result: list[tuple[np.ndarray, int, str, str]] = []
    for start, end, cid in segments:
        for window, wcid in cut_windows_from_segment(y, start, end, cid):
            result.append((window, wcid, file_id, player_id))

    return result


# Process all files
audio_files = sorted(AUDIO_DIR.glob("*_mic.wav"))
print(f"Processing {len(audio_files)} audio files...")

all_examples: list[tuple[np.ndarray, int, str, str]] = []
for audio_path in tqdm(audio_files, desc="Files"):
    all_examples.extend(process_one_file(audio_path))

print(f"\nTotal training examples: {len(all_examples)}")

# Overall class distribution
from collections import Counter
overall_counts = Counter(ID_TO_CLASS[cid] for _, cid, _, _ in all_examples)
print(f"\nClasses present: {len(overall_counts)}/{NUM_CLASSES}")
print("\nTop 15 most common classes:")
for name, cnt in overall_counts.most_common(15):
    print(f"  {name}: {cnt}")

print("\nBottom 10 least common classes:")
for name, cnt in overall_counts.most_common()[-10:]:
    print(f"  {name}: {cnt}")

Processing 360 audio files...


Files: 100%|██████████| 360/360 [02:11<00:00,  2.74it/s]


Total training examples: 19667

Classes present: 36/37

Top 15 most common classes:
  maj:C#: 1698
  maj:D#: 1362
  maj:C: 1332
  maj:E: 1248
  maj:F: 1224
  maj:G#: 1212
  maj:F#: 1182
  maj:A: 1092
  maj:D: 1002
  maj:G: 834
  maj:A#: 810
  maj:B: 720
  min:F: 576
  min:G: 516
  min:A#: 504

Bottom 10 least common classes:
  note:C#: 159
  min:C: 156
  note:D#: 155
  note:G: 142
  note:C: 141
  note:E: 138
  note:F#: 132
  note:D: 128
  note:B: 113
  note:A: 105


In [8]:
def extract_silence_from_solo(
    jam: jams.JAMS,
    audio: np.ndarray,
    max_segments_per_file: int = 3,
    min_gap_sec: float = 0.6,
) -> list[tuple[np.ndarray, int]]:
    """
    Find gaps in solo playing (no notes on any string) and extract silence windows.
    """
    note_anns = jam.search(namespace="note_midi")
    if not note_anns:
        return []

    # Collect all note time ranges across all strings
    note_ranges: list[tuple[float, float]] = []
    for ann_raw in note_anns:
        ann = cast(Annotation, ann_raw)
        for obs in ann.data:
            start = float(obs.time)
            note_ranges.append((start, start + float(obs.duration)))

    if not note_ranges:
        return []

    # Sort by start time and merge overlapping ranges
    note_ranges.sort()
    merged: list[tuple[float, float]] = [note_ranges[0]]
    for start, end in note_ranges[1:]:
        last_start, last_end = merged[-1]
        if start <= last_end:
            merged[-1] = (last_start, max(last_end, end))
        else:
            merged.append((start, end))

    # Find gaps between merged ranges
    duration_sec = len(audio) / SR
    gaps: list[tuple[float, float]] = []
    # Gap before first note
    if merged[0][0] > min_gap_sec:
        gaps.append((0.0, merged[0][0]))
    # Gaps between notes
    for i in range(len(merged) - 1):
        gap_start = merged[i][1]
        gap_end = merged[i + 1][0]
        if gap_end - gap_start > min_gap_sec:
            gaps.append((gap_start, gap_end))
    # Gap after last note
    if duration_sec - merged[-1][1] > min_gap_sec:
        gaps.append((merged[-1][1], duration_sec))

    # Cut silence windows from the gaps (up to max_segments_per_file)
    silence_windows: list[tuple[np.ndarray, int]] = []
    silence_cid = CLASS_TO_ID["silence"]

    for gap_start, gap_end in gaps[:max_segments_per_file]:
        # Take the middle of the gap for cleanest silence
        gap_center = (gap_start + gap_end) / 2
        win_start = gap_center - WINDOW_SEC / 2
        if win_start < 0:
            continue
        start_sample = int(win_start * SR)
        end_sample = start_sample + WINDOW_SAMPLES
        if end_sample > len(audio):
            continue
        silence_windows.append((audio[start_sample:end_sample].copy(), silence_cid))

    return silence_windows


# Run silence extraction over all solo files
solo_files = [p for p in audio_files if "_solo" in p.name]
print(f"Extracting silence from {len(solo_files)} solo files...")

silence_examples: list[tuple[np.ndarray, int, str, str]] = []
for audio_path in tqdm(solo_files, desc="Silence"):
    file_id, player_id, _ = parse_file_id(audio_path.name)
    anno_path = ANNO_DIR / f"{file_id}.jams"
    if not anno_path.exists():
        continue
    y, _ = librosa.load(audio_path, sr=SR, mono=True)
    jam = jams.load(str(anno_path))
    for window, cid in extract_silence_from_solo(jam, y):
        silence_examples.append((window, cid, file_id, player_id))

print(f"\nExtracted {len(silence_examples)} silence examples")

# Add to main dataset
all_examples.extend(silence_examples)
print(f"Total examples now: {len(all_examples)}")

# Refresh distribution
overall_counts = Counter(ID_TO_CLASS[cid] for _, cid, _, _ in all_examples)
print(f"Classes present: {len(overall_counts)}/{NUM_CLASSES}")
print(f"Silence examples: {overall_counts.get('silence', 0)}")

Extracting silence from 180 solo files...


Silence: 100%|██████████| 180/180 [00:34<00:00,  5.22it/s]


Extracted 316 silence examples
Total examples now: 19983
Classes present: 37/37
Silence examples: 316


In [9]:
import pandas as pd

# Stack all audio windows into one array
X_audio = np.stack([window for window, _, _, _ in all_examples], axis=0)
print(f"X_audio shape: {X_audio.shape}, dtype: {X_audio.dtype}")
print(f"X_audio size on disk (est.): {X_audio.nbytes / 1e6:.1f} MB")

# Metadata DataFrame — everything except the audio itself
metadata_df = pd.DataFrame({
    "class_id": [cid for _, cid, _, _ in all_examples],
    "class_name": [ID_TO_CLASS[cid] for _, cid, _, _ in all_examples],
    "file_id": [fid for _, _, fid, _ in all_examples],
    "player_id": [pid for _, _, _, pid in all_examples],
})

print(f"\nMetadata shape: {metadata_df.shape}")
print(metadata_df.head())

# Save both
audio_path_out = PROCESSED_DIR / "windows_audio.npy"
metadata_path_out = PROCESSED_DIR / "windows_metadata.parquet"

np.save(audio_path_out, X_audio)
metadata_df.to_parquet(metadata_path_out, index=False)

print(f"\nSaved:")
print(f"  {audio_path_out} ({audio_path_out.stat().st_size / 1e6:.1f} MB)")
print(f"  {metadata_path_out} ({metadata_path_out.stat().st_size / 1e6:.2f} MB)")

X_audio shape: (19983, 11025), dtype: float32
X_audio size on disk (est.): 881.3 MB

Metadata shape: (19983, 4)
   class_id class_name             file_id player_id
0        15     maj:D#  00_BN1-129-Eb_comp        00
1        15     maj:D#  00_BN1-129-Eb_comp        00
2        15     maj:D#  00_BN1-129-Eb_comp        00
3        15     maj:D#  00_BN1-129-Eb_comp        00
4        15     maj:D#  00_BN1-129-Eb_comp        00

Saved:
  c:\Users\Constantine\Code\ml-project\data\processed\windows_audio.npy (881.3 MB)
  c:\Users\Constantine\Code\ml-project\data\processed\windows_metadata.parquet (0.02 MB)


In [10]:
# Split by player — the critical decision
TRAIN_PLAYERS = {"00", "01", "02", "03"}
VAL_PLAYERS = {"04"}
TEST_PLAYERS = {"05"}

train_mask = metadata_df["player_id"].isin(TRAIN_PLAYERS)
val_mask = metadata_df["player_id"].isin(VAL_PLAYERS)
test_mask = metadata_df["player_id"].isin(TEST_PLAYERS)

print(f"Train: {train_mask.sum()} examples ({train_mask.mean() * 100:.1f}%)")
print(f"Val:   {val_mask.sum()} examples ({val_mask.mean() * 100:.1f}%)")
print(f"Test:  {test_mask.sum()} examples ({test_mask.mean() * 100:.1f}%)")

# Save split indices as a column
metadata_df["split"] = "train"
metadata_df.loc[val_mask, "split"] = "val"
metadata_df.loc[test_mask, "split"] = "test"

# Re-save metadata with the split column
metadata_df.to_parquet(metadata_path_out, index=False)

# Class coverage check — every split should see every class
print(f"\nClass coverage per split:")
for split_name in ("train", "val", "test"):
    split_classes = metadata_df.loc[metadata_df["split"] == split_name, "class_id"].nunique()
    print(f"  {split_name}: {split_classes}/{NUM_CLASSES} classes present")

# If validation or test is missing classes, we'll know now
missing_in_val = set(range(NUM_CLASSES)) - set(metadata_df.loc[val_mask, "class_id"].unique())
missing_in_test = set(range(NUM_CLASSES)) - set(metadata_df.loc[test_mask, "class_id"].unique())
if missing_in_val:
    print(f"\n  Classes missing from val: {[ID_TO_CLASS[c] for c in sorted(missing_in_val)]}")
if missing_in_test:
    print(f"  Classes missing from test: {[ID_TO_CLASS[c] for c in sorted(missing_in_test)]}")

Train: 12964 examples (64.9%)
Val:   3513 examples (17.6%)
Test:  3506 examples (17.5%)

Class coverage per split:
  train: 37/37 classes present
  val: 37/37 classes present
  test: 37/37 classes present
